In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import hvplot.pandas  # Adds .hvplot() to our tables and maps.
import holoviews as hv
import rasterio
from rasterio.mask import mask
from rasterstats import zonal_stats
from IPython.display import display

%matplotlib inline
hv.extension("bokeh")
pd.options.display.max_columns = 999
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

# Try the usual data folder, then your assignment3-data folder.
# The last option also lets the copy in the Codex outputs folder run here.
data_folders = [
    Path("data"),
    Path("assignment3-data"),
    Path("C:/Users/leung/Documents/MUSA_5500_Python/Assignment-3/assignment3-data"),
]
for folder in data_folders:
    if (folder / "PA-tracts.geojson").exists():
        DATA = folder
        break
else:
    raise FileNotFoundError("Put the six assignment data files in a folder named data.")

print("Data folder:", DATA)

# Assignment 3

This assignment will contain two parts:

1. Exploring evictions and code violations in Philadelphia
2. Comparing the NDVI in Philadelphia


Download all the necessary datasets from [here](assignment3-data.zip) before starting the assignment. Please put the datasets in a folder named `data` in your working directory with your `.ipynb` file.

## Part 1: Exploring Evictions and Code Violations in Philadelphia

In this assignment, we'll explore spatial trends evictions in Philadelphia using data from the [Eviction Lab](https://evictionlab.org/) and building code violations using data from [OpenDataPhilly](https://www.opendataphilly.org/).

We'll be exploring the idea that evictions can occur as retaliation against renters for reporting code violations. Spatial correlations between evictions and code violations from the City's Licenses and Inspections department can offer some insight into this question. 

**A couple of interesting background readings:**
- [HuffPost article](https://www.huffingtonpost.com/entry/cities-are-starting-to-pay-attention-to-the-eviction-crisis-thats-devastated-poor-tenants_us_5b1a7b21e4b0bbb7a0dbd59e)
- [PlanPhilly article](http://planphilly.com/articles/2018/04/12/philly-landlords-evict-more-people-than-owners-in-other-large-cities)

## 1.1 Explore Eviction Lab Data

The Eviction Lab built the first national database for evictions. If you aren't familiar with the project, you can explore their website: https://evictionlab.org/ 

### 1.1.1 Read data using `geopandas`

The first step is to read the eviction data by census tract using `geopandas`. The data for all of Pennsylvania by census tract is available in the `data/` folder in a GeoJSON format.

Load the data file "PA-tracts.geojson" using `geopandas`

**Note:** If you'd like to see all columns in the data frame, you can increase the max number of columns using pandas display options:

**In plain English:** A census tract is a small area used for census statistics. Each row describes one tract; `geometry` stores its outline.

In [ ]:
# A GeoDataFrame is a table with an extra column that stores map shapes.
pa_tracts = gpd.read_file(DATA / "PA-tracts.geojson")
print(f"Pennsylvania census tracts: {len(pa_tracts):,}")
pa_tracts[["GEOID", "pl", "geometry"]].head(3)

In [ ]:
# Preview the columns we need. In the dictionary, pl means parent location.
pa_tracts[["GEOID", "n", "pl", "e-03", "e-16", "geometry"]].head()

### 1.1.2 Explore and trim the data 

We will need to trim data to Philadelphia only. Take a look at the data dictionary for the descriptions of the various columns in top-level repository folder: `eviction_lab_data_dictionary.txt`

**Note:** the column names are shortened — see the end of the above file for the abbreviations. The numbers at the end of the columns indicate the years. For example, `e-16` is the number of evictions in 2016. 

Take a look at the individual columns and trim to census tracts in Philadelphia. (**Hint:** Philadelphia is both a city and a county).

**Why this filter?** Philadelphia is both a city and a county. The data dictionary says `pl` identifies the county and `e-16` counts eviction judgments in 2016, rather than all court filings.

In [ ]:
# Keep only tracts whose parent county is Philadelphia.
philly_tracts = pa_tracts.loc[
    pa_tracts["pl"] == "Philadelphia County, Pennsylvania"
].copy()

# Treat a geographic ID as a label, not a number to calculate with.
philly_tracts["GEOID"] = philly_tracts["GEOID"].astype(str)
print(f"Philadelphia census tracts: {len(philly_tracts):,}")
philly_tracts[["GEOID", "pl", "e-16"]].head()

### 1.1.3 Transform from wide to tidy format

For this assignment, we are interested in the number of evictions by census tract for various years. Right now, each year has it's own column, so it will be easiest to transform to a tidy format. 

Use the `pd.melt()` function to transform the eviction data into tidy format, using the number of evictions from **2003 to 2016**.

The tidy data frame should have four columns: `GEOID`, `geometry`, a column holding the number of evictions, and a column telling you what the name of the original column was for that value.

**Hints:**
- You'll want to specify the `GEOID` and `geometry` columns as the `id_vars`. This will keep track of the census tract information. 
- You should specify the names of the columns holding the number of evictions as the `value_vars`.
- You can generate a list of this column names using Python's f-string formatting:
    ```python
    value_vars = [f"e-{x:02d}" for x in range(3, 17)]
    ```

**Wide versus tidy:** Wide data has one column per year. Tidy data has one row per tract per year. Think of stacking the year-columns underneath one another. The result has exactly four columns.

In [ ]:
# Make the list e-03, e-04, ..., e-16.
eviction_columns = [f"e-{year:02d}" for year in range(3, 17)]

# Melt turns the 14 year-columns into rows, keeping the tract ID and shape.
evictions_tidy = pd.melt(
    philly_tracts,
    id_vars=["GEOID", "geometry"],
    value_vars=eviction_columns,
    var_name="eviction_year",
    value_name="evictions",
)
# Explicitly keep the map information after reshaping the table.
evictions_tidy = gpd.GeoDataFrame(
    evictions_tidy, geometry="geometry", crs=philly_tracts.crs
)
print(f"Rows: {len(evictions_tidy):,} = {len(philly_tracts)} tracts x 14 years")
evictions_tidy.head()

### 1.1.4 Plot the total number of evictions per year from 2003 to 2016

Use `hvplot` to plot the total number of evictions from 2003 to 2016. You will first need to perform a group by operation and sum up the total number of evictions for all census tracts, and then use `hvplot()` to make your plot.

You can use any type of `hvplot` chart you'd like to show the trend in number of evictions over time.

**What `groupby` does:** Put all rows from the same year into a pile, then add the eviction counts in each pile.

In [ ]:
# Add the tract counts for each year. min_count=1 keeps all-missing totals missing.
annual_evictions = (
    evictions_tidy.groupby("eviction_year")["evictions"]
    .sum(min_count=1)
    .reset_index()
)
# Turn a label such as e-03 into the year 2003.
annual_evictions["year"] = 2000 + annual_evictions["eviction_year"].str[2:].astype(int)
annual_evictions = annual_evictions.sort_values("year")
display(annual_evictions[["year", "evictions"]])

annual_evictions.hvplot.line(
    x="year", y="evictions", width=760, height=360,
    color="#176b87", line_width=3,
    title="Philadelphia eviction judgments, 2003–2016",
    xlabel="Year", ylabel="Total evictions", ylim=(0, 12000),
)

**Reading the result:** Recorded evictions stay near 10,000–11,000 per year. The highest total is 11,182 in 2014; 2016 has 10,264. These are counts, so this chart does not adjust for changes in the number of renter households.

### 1.1.5 The number of evictions across Philadelphia

Our tidy data frame is still a GeoDataFrame with a geometry column, so we can visualize the number of evictions for all census tracts. 

Use `hvplot()` to generate a choropleth showing the number of evictions for a specified year, with a widget dropdown to select a given year (or variable name, e.g., `e-16`, `e-15`, etc). 

**Hints** 
- You'll need to use the `groupby` keyword to tell `hvplot` to make a series of maps, with a widget to select between them.
- You will need to specify `dynamic=False` as a keyword argument to the `hvplot()` function. 
- Be sure to specify a `width` and `height` that makes your output map (roughly) square to limit distortions

**How to use this map:** Choose a year from the dropdown and hover over a tract to see its count. A choropleth colors areas by a number. The color scale stays fixed across years, so the same color means the same count.

In [ ]:
# A projected CRS gives us a sensible city map instead of stretched longitude/latitude.
evictions_map_data = evictions_tidy.to_crs(epsg=3857)

eviction_maps = evictions_map_data.hvplot.polygons(
    c="evictions", groupby="eviction_year", dynamic=False,
    geo=True, crs=3857, width=620, height=620,
    cmap="YlOrRd", colorbar=True,
    clim=(0, evictions_tidy["evictions"].max()),
    hover_cols=["GEOID", "evictions"],
    line_color="white", line_width=0.25,
    title="Evictions by census tract", clabel="Evictions",
    xaxis=None, yaxis=None,
)
eviction_maps

## 1.2 Code Violations in Philadelphia

Next, we'll explore data for code violations from the Licenses and Inspections Department of Philadelphia to look for potential correlations with the number of evictions.


### 1.2.1 Load data from 2012 to 2016

L+I violation data for years including 2012 through 2016 (inclusive) is provided in a CSV format in the "data/" folder. 

Load the data using pandas and convert to a GeoDataFrame.

**Data note:** The assignment describes this file as covering 2012–2016. It contains `lat`, `lng`, and `violationdescription`, but no date column, so we cannot independently filter or check individual years. We count rows as violation records, not unique buildings.

In [ ]:
violations_df = pd.read_csv(DATA / "li_violations.csv")
print(f"Rows in the supplied file: {len(violations_df):,}")
print("Columns:", violations_df.columns.tolist())

# A point needs both longitude (left/right) and latitude (up/down).
valid_coordinates = (
    violations_df["lng"].between(-180, 180)
    & violations_df["lat"].between(-90, 90)
)
print(f"Rows without usable coordinates: {(~valid_coordinates).sum():,}")
violations_df = violations_df.loc[valid_coordinates].copy()
violations = gpd.GeoDataFrame(
    violations_df,
    geometry=gpd.points_from_xy(violations_df["lng"], violations_df["lat"]),
    crs="EPSG:4326",
)
violations.head()

### 1.2.2 Trim to specific violation types

There are many different types of code violations (running the `nunique()` function on the `violationdescription` column will extract all of the unique ones). More information on different types of violations can be found [on the City's website](https://www.phila.gov/li/codesandregulations/Pages/codes.aspx).

Below, I've selected 15 types of violations that deal with property maintenance and licensing issues. We'll focus on these violations. The goal is to see if these kinds of violations are correlated spatially with the number of evictions in a given area. 

Use the list of violations given to trim your data set to only include these types.

In [ ]:
violation_types = [
    "INT-PLMBG MAINT FIXTURES-RES",
    "INT S-CEILING REPAIR/MAINT SAN",
    "PLUMBING SYSTEMS-GENERAL",
    "CO DETECTOR NEEDED",
    "INTERIOR SURFACES",
    "EXT S-ROOF REPAIR",
    "ELEC-RECEPTABLE DEFECTIVE-RES",
    "INT S-FLOOR REPAIR",
    "DRAINAGE-MAIN DRAIN REPAIR-RES",
    "DRAINAGE-DOWNSPOUT REPR/REPLC",
    "LIGHT FIXTURE DEFECTIVE-RES",
    "LICENSE-RES SFD/2FD",
    "ELECTRICAL -HAZARD",
    "VACANT PROPERTIES-GENERAL",
    "INT-PLMBG FIXTURES-RES",
]

**What `isin` means:** Keep a row if its description appears in the supplied list. `nunique()` counts distinct descriptions; it does not list their names.

In [ ]:
print("Different named violation types:", violations["violationdescription"].nunique())
selected_violations = violations.loc[
    violations["violationdescription"].isin(violation_types)
].copy()
print(f"Records in the 15 selected types: {len(selected_violations):,}")
selected_violations["violationdescription"].value_counts()

### 1.2.3 Make a hex bin map

The code violation data is **point data**. We can get a quick look at the geographic distribution using `matplotlib` and the `hexbin()` function. Make a hex bin map of the code violations and overlay the census tract outlines. 

**Hints**:
- The eviction data from part 1 was by census tract, so the census tract geometries are available as part of that GeoDataFrame. You can use it to overlay the census tracts on your hex bin map.
- Make sure you convert your GeoDataFrame to a CRS that's better for visualization than plain old 4326.

**A hex bin map** divides the map into little hexagons and counts the points in each one. The thin outlines show census tracts. A CRS is a coordinate system: both layers use EPSG:3857 here so their positions line up.

In [ ]:
selected_3857 = selected_violations.to_crs(epsg=3857)
tracts_3857 = philly_tracts[["GEOID", "geometry"]].to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(8, 8))
hexagons = ax.hexbin(
    selected_3857.geometry.x, selected_3857.geometry.y,
    gridsize=45, mincnt=1, cmap="magma_r",
)
tracts_3857.boundary.plot(ax=ax, color="#666666", linewidth=0.35)
fig.colorbar(hexagons, ax=ax, shrink=0.72, label="Violation records per hexagon")
ax.set_title("Selected code violations, 2012–2016")
ax.set_aspect("equal")
ax.set_axis_off()
plt.tight_layout()
plt.show()

### 1.2.4 Spatially join data sets

To do a census tract comparison to our eviction data, we need to find which census tract each of the code violations falls into. Use the `geopandas.sjoin()` function to do just that. 


**Hints**
- You can re-use your eviction data frame, but you will only need the `geometry` column (specifying census tract polygons) and the `GEOID` column (specifying the name of each census tract).
- Make sure both data frames have the same CRS before joining them together!

**Spatial join = match by location.** A normal table join matches IDs; this join matches each dot to the outline containing it. A left join keeps unmatched points so we can notice them. Points exactly on a boundary would not match `within`.

In [ ]:
# A spatial join asks: which tract contains each violation point?
tract_shapes = philly_tracts[["GEOID", "geometry"]].copy()
violations_with_tract = gpd.sjoin(
    selected_violations.to_crs(tract_shapes.crs),
    tract_shapes,
    how="left", predicate="within",
)
print("Points not matched to a tract:", violations_with_tract["GEOID"].isna().sum())
violations_with_tract[["violationdescription", "GEOID"]].head()

### 1.2.5 Calculate the number of violations by type per census tract

Next, we'll want to find the number of violations (for each kind) per census tract. You should group the data frame by violation type and census tract name.

The result of this step should be a data frame with three columns: `violationdescription`, `GEOID`, and `N`, where `N` is the number of violations of that kind in the specified census tract.

**Optional: to make prettier plots**

Some census tracts won't have any violations, and they won't be included when we do the above calculation. However, there is a trick to set the values for those census tracts to be zero. After you calculate the sizes of each violation/census tract group, you can run: 

```python
N = N.unstack(fill_value=0).stack().reset_index(name='N')
```
where `N` gives the total size of each of the groups, specified by violation type and census tract name.

See [this StackOverflow post](https://stackoverflow.com/questions/42854801/including-missing-combinations-of-values-in-a-pandas-groupby-aggregation) for more details.

This part is **optional**, but will make the resulting maps a bit prettier.

**Why fill in zeros?** A tract with no recorded violations should appear with zero, rather than disappear from the map. `N` means the number of records of one type in one tract.

In [ ]:
# Count the rows for every combination of violation type and tract.
counts = violations_with_tract.groupby(["violationdescription", "GEOID"]).size()

# Include all 15 types and all tracts, even when a combination has zero records.
all_pairs = pd.MultiIndex.from_product(
    [violation_types, tract_shapes["GEOID"]],
    names=["violationdescription", "GEOID"],
)
violation_counts = counts.reindex(all_pairs, fill_value=0).reset_index(name="N")
violation_counts.head()

### 1.2.6 Merge with census tracts geometries

We now have the number of violations of different types per census tract specified as a regular DataFrame. You can now merge it with the census tract geometries (from your eviction data GeoDataFrame) to create a GeoDataFrame. 

**Hints**
- Use `pandas.merge()` and specify the `on` keyword to be the column holding census tract names. 
- Make sure the result of the merge operation is a GeoDataFrame — you will want the GeoDataFrame holding census tract geometries to be the first argument of the `pandas.merge()` function.

**Merge = match by ID.** Keeping the GeoDataFrame on the left preserves the geometry and CRS. Each tract now appears once for each violation type.

In [ ]:
# Attach the tract outlines to the counts using their shared GEOID labels.
violations_by_tract = pd.merge(tract_shapes, violation_counts, on="GEOID")
print("Result type:", type(violations_by_tract).__name__)
violations_by_tract.head()

### 1.2.7 Interactive choropleths for each violation type

Now, we can use `hvplot()` to create an interactive choropleth for each violation type and add a widget to specify different violation types. 

**Hints** 
- You'll need to use the `groupby` keyword to tell `hvplot` to make a series of maps, with a widget to select different violation types.
- You will need to specify `dynamic=False` as a keyword argument to the `hvplot()` function. 
- Be sure to specify a `width` and `height` that makes your output map (roughly) square to limit distortions

**Try the dropdown:** Switch among the 15 types. Hover to read exact counts. The color scale stays the same across these maps, so less common types may look pale.

In [ ]:
violations_map_data = violations_by_tract.to_crs(epsg=3857)
violation_maps = violations_map_data.hvplot.polygons(
    c="N", groupby="violationdescription", dynamic=False,
    geo=True, crs=3857, width=620, height=620,
    cmap="Blues", colorbar=True, clim=(0, violation_counts["N"].max()),
    hover_cols=["GEOID", "N"], line_color="white", line_width=0.25,
    title="Selected violations by census tract, 2012–2016",
    clabel="Violation records", xaxis=None, yaxis=None,
)
violation_maps

## 1.3. A side-by-side comparison

From the interactive maps of evictions and violations, you should notice a lot of spatial overlap. 

As a final step, we'll make a side-by-side comparison to better show the spatial correlations. This will involve a few steps: 

1. Trim the evictions data frame plotted in section 1.1.5 to only include evictions from 2016. 
2. Trim the L+I violations data frame plotted in section 1.2.7 to only include a single violation type (pick whichever one you want!).
3. Use `hvplot()` to make two interactive choropleth maps, one for the data from step 1. and one for the data in step 2.
4. Show these two plots side by side (one row and 2 columns) using the syntax for combining charts.

**Note:** since we selected a single year and violation type, you won't need to use the `groupby=` keyword here.

**Compare locations, then read each legend:** These maps use separate scales. One shows a single year of evictions; the other shows five years of ceiling-maintenance violations.

In [ ]:
evictions_2016 = evictions_map_data.loc[
    evictions_map_data["eviction_year"] == "e-16"
].copy()
chosen_type = "INT S-CEILING REPAIR/MAINT SAN"
chosen_violations = violations_map_data.loc[
    violations_map_data["violationdescription"] == chosen_type
].copy()

eviction_plot = evictions_2016.hvplot.polygons(
    c="evictions", geo=True, crs=3857, width=440, height=470,
    cmap="YlOrRd", colorbar=True, clim=(0, evictions_2016["evictions"].max()),
    hover_cols=["GEOID"], line_color="white", line_width=0.25,
    title="Evictions in 2016", clabel="Evictions", xaxis=None, yaxis=None,
)
violation_plot = chosen_violations.hvplot.polygons(
    c="N", geo=True, crs=3857, width=440, height=470,
    cmap="Blues", colorbar=True, clim=(0, chosen_violations["N"].max()),
    hover_cols=["GEOID"], line_color="white", line_width=0.25,
    title="Ceiling violations, 2012–2016", clabel="Records", xaxis=None, yaxis=None,
)

# + puts charts beside each other; cols(2) requests two columns.
(eviction_plot + violation_plot).cols(2)

**What this can tell us:** Some high-count areas overlap, especially in North and West Philadelphia. That is a geographic association, not proof that reporting a violation caused an eviction. The records do not link individual complaints to later eviction cases, and counts can also be higher where there are more rental homes.

## 1.4. Extra Credit

Identify the 20 most common types of violations within the time period of 2012 to 2016 and create a set of interactive choropleths similar to what was done in section 1.2.7. 

Use this set of maps to identify 3 types of violations that don't seem to have much spatial overlap with the number of evictions in the City.

**Extra credit:** These maps include the 20 most frequent descriptions in the full file. Here the color scale adjusts when you change the type, making each pattern easier to see. Read the legend each time; colors are not comparable across types.

In [ ]:
# Extra credit starts from ALL violation types, not just the selected 15.
top20_counts = violations["violationdescription"].value_counts().head(20)
display(top20_counts.rename_axis("violationdescription").reset_index(name="records"))
top20_points = violations.loc[violations["violationdescription"].isin(top20_counts.index)]
top20_joined = gpd.sjoin(
    top20_points.to_crs(tract_shapes.crs), tract_shapes,
    how="left", predicate="within",
)
print("Unmatched top-20 records:", top20_joined["GEOID"].isna().sum())
top20_pairs = pd.MultiIndex.from_product(
    [top20_counts.index, tract_shapes["GEOID"]],
    names=["violationdescription", "GEOID"],
)
top20_by_tract = (
    top20_joined.groupby(["violationdescription", "GEOID"]).size()
    .reindex(top20_pairs, fill_value=0).reset_index(name="N")
)
top20_map_data = pd.merge(tract_shapes, top20_by_tract, on="GEOID").to_crs(epsg=3857)
top20_maps = top20_map_data.hvplot.polygons(
    c="N", groupby="violationdescription", dynamic=False,
    geo=True, crs=3857, width=620, height=620,
    cmap="Purples", colorbar=True, hover_cols=["GEOID", "N"],
    line_color="white", line_width=0.25,
    title="20 most common violation types, 2012–2016", clabel="Records",
    xaxis=None, yaxis=None,
).opts(framewise=True)
top20_maps

**Finding three weaker matches:** Pearson correlation summarizes how closely two sets of tract counts rise together. Values near +1 indicate a strong positive relationship; values near 0 indicate a weak linear relationship. Negative values indicate opposite tendencies. We use it to support the map comparison, not as a test of retaliation.

In [ ]:
# Make one column per type so we can compare each with 2016 evictions.
top20_wide = top20_by_tract.pivot(index="GEOID", columns="violationdescription", values="N")
evictions_for_comparison = philly_tracts.set_index("GEOID")["e-16"]
correlations = top20_wide.corrwith(evictions_for_comparison)
weakest_types = correlations.abs().sort_values().head(3).index
display(correlations.loc[weakest_types].rename("Pearson r").to_frame().round(3))

# Show the three selected patterns with the eviction map for a direct comparison.
weak_maps = []
for violation_type in weakest_types:
    one_type = top20_map_data.loc[top20_map_data["violationdescription"] == violation_type]
    one_map = one_type.hvplot.polygons(
        c="N", geo=True, crs=3857, width=440, height=470,
        cmap="Purples", colorbar=True, clim=(0, one_type["N"].max()),
        hover_cols=["GEOID", "N"], line_color="white", line_width=0.25,
        title=violation_type, clabel="Records", xaxis=None, yaxis=None,
    )
    weak_maps.append(one_map)
(eviction_plot + weak_maps[0] + weak_maps[1] + weak_maps[2]).cols(2)

**Three types with relatively little overlap with 2016 evictions:**

- `VIOL C&I MESSAGE` (r = −0.127): its strongest concentrations do not line up well with the high-eviction tracts.
- `LICENSE-RES GENERAL` (r = 0.157): its strongest clusters differ from the main eviction clusters.
- `EXT A-VACANT LOT CLEAN/MAINTAI` (r = 0.209): some areas overlap, but the pattern is more concentrated in particular tracts than the eviction pattern.

These are the three smallest absolute correlations among the top 20 types. “Little overlap” is relative: it does not mean the maps share no locations. The maps compare 2012–2016 violation counts with 2016 eviction counts, without adjusting for tract population or rental housing.

## Part 2: Exploring the NDVI in Philadelphia

In this part, we'll explore the NDVI in Philadelphia a bit more. This part will include two parts:

1. We'll compare the median NDVI within the city limits and the immediate suburbs
1. We'll calculate the  NDVI around street trees in the city.

## 2.1 Comparing the NDVI in the city and the suburbs

### 2.1.1 Load Landsat data for Philadelphia

Use rasterio to load the landsat data for Philadelphia (available in the "data/" folder)

**A raster is a grid of pixels**, like a digital photograph. A band is one layer measuring a particular kind of light. This file has 30-meter pixels; the course uses band 4 for red light and band 5 for near-infrared light.

In [ ]:
landsat_path = DATA / "landsat8_philly.tif"
with rasterio.open(landsat_path) as landsat:
    raster_crs = landsat.crs
    raster_transform = landsat.transform
    print("CRS:", raster_crs)
    print("Number of bands:", landsat.count)
    print("Image size (rows, columns):", landsat.shape)
    print("Pixel size in meters:", landsat.res)

### 2.1.2 Separating the city from the suburbs

Create two polygon objects, one for the city limits and one for the suburbs. To calculate the suburbs polygon, we will take everything outside the city limits but still within the bounding box. 

* The city limits are available in the "data/" folder.
* To calculate the suburbs polygon, the "envelope" attribute of the city limits geometry will be useful.
* You can use geopandas' [geometric manipulation](https://geopandas.org/en/stable/docs/user_guide/geometric_manipulations.html) functionality to calculate the suburbs polygon from the city limits polygon and the envelope polygon.

**What counts as “suburbs” here?** Only the area outside the city but inside its bounding rectangle, as the assignment specifies. This is not the whole metropolitan area and may include water. We create the rectangle after projecting into the raster CRS.

In [ ]:
# Convert the boundary to the same coordinates as the satellite image.
city_limits = gpd.read_file(DATA / "City_Limits.geojson").to_crs(raster_crs)
city_polygon = city_limits.geometry.iloc[0]

# envelope makes a rectangle around the city; difference cuts the city out.
city_rectangle = city_polygon.envelope
suburbs_polygon = city_rectangle.difference(city_polygon)

regions = gpd.GeoDataFrame(
    {"region": ["City", "Surrounding area inside rectangle"]},
    geometry=[city_polygon, suburbs_polygon], crs=raster_crs,
)
ax = regions.plot(column="region", categorical=True, legend=True,
                  cmap="Set2", edgecolor="white", figsize=(7, 7))
city_limits.boundary.plot(ax=ax, color="black", linewidth=0.7)
ax.set_title("City and the assignment's surrounding area")
ax.set_axis_off()
plt.tight_layout()
plt.show()

### 2.1.3 Mask and calculate the NDVI for the city and the suburbs

Using the two polygons from the last section, use rasterio's mask functionality to create two masked arrays from the landsat data, one for the city and one for the suburbs. 

For each masked array, calculate the NDVI.

**NDVI = (near-infrared − red) ÷ (near-infrared + red).** Plants tend to reflect more near-infrared light and absorb red light, so larger NDVI values generally suggest more vegetation. `NaN` means “missing”; pixels outside each polygon stay missing rather than becoming zero.

This follows the course's calculation on the supplied bands. The file does not include acquisition-date, calibration, or cloud-quality information, so treat the results as an exercise using this image, not a calibrated assessment of current vegetation. The comparison uses only pixels covered by the supplied raster.

In [ ]:
# Keep pixels whose centers lie inside each polygon; mask everything else.
with rasterio.open(landsat_path) as landsat:
    city_bands, city_transform = mask(
        landsat, [city_polygon], indexes=[4, 5], crop=True,
        filled=False, all_touched=False,
    )
    suburb_bands, suburb_transform = mask(
        landsat, [suburbs_polygon], indexes=[4, 5], crop=True,
        filled=False, all_touched=False,
    )

def calculate_ndvi(red, nir):
    """Compare near-infrared light with red light; leave missing pixels as NaN."""
    # Floats allow negative answers and prevent unsigned-integer overflow.
    red = red.astype(float).filled(np.nan)
    nir = nir.astype(float).filled(np.nan)
    denominator = nir + red
    denominator[denominator == 0] = np.nan  # Cannot divide by zero.
    return (nir - red) / denominator

# We read only bands [4, 5], so array index 0 is red and index 1 is NIR.
city_ndvi = calculate_ndvi(city_bands[0], city_bands[1])
suburb_ndvi = calculate_ndvi(suburb_bands[0], suburb_bands[1])
print("Usable city pixels:", np.isfinite(city_ndvi).sum())
print("Usable surrounding-area pixels:", np.isfinite(suburb_ndvi).sum())

### 2.1.4 Calculate the median NDVI within the city and within the suburbs

* Calculate the median value from your NDVI arrays for the city and suburbs
* Numpy's `nanmedian` function will be useful for ignoring NaN elements
* Print out the median values. Which has a higher NDVI: the city or suburbs?

In [ ]:
# The median is the middle value after sorting. nanmedian skips missing pixels.
city_median = np.nanmedian(city_ndvi)
suburb_median = np.nanmedian(suburb_ndvi)
print(f"City median NDVI: {city_median:.3f}")
print(f"Surrounding-area median NDVI: {suburb_median:.3f}")

if suburb_median > city_median:
    print("The surrounding area has the higher median NDVI in this image.")
else:
    print("The city has an equal or higher median NDVI in this image.")
print(f"Difference (surrounding area minus city): {suburb_median - city_median:.3f}")

## 2.2 Calculating the NDVI for Philadelphia's street trees

### 2.2.1 Load the street tree data

The data is available in the "data/" folder. It has been downloaded from [OpenDataPhilly](https://www.opendataphilly.org/dataset/ppr-tree-canopy). It contains the locations of abot 2,500 street trees in Philadelphia.

**Match coordinates first:** The tree points and raster must use the same CRS. Otherwise the computer would look up pixels in the wrong places.

In [ ]:
street_trees = gpd.read_file(DATA / "ppr_tree_canopy_points_2015.geojson")
street_trees = street_trees.to_crs(raster_crs)
print(f"Street tree points: {len(street_trees):,}")
street_trees.head()

### 2.2.2 Calculate the NDVI values at the locations of the street trees

* Use the rasterstats package to calculate the NDVI values at the locations of the street trees.
* Since these are point geometries, you can calculate either the median or the mean statistic (only one pixel will contain each point).

**What the transform does:** It connects a raster row and column to a real map location. Use the full raster transform with the full array; a cropped array would need its own transform. We sample the pixel at each point, without adding a buffer around the tree.

In [ ]:
# Calculate NDVI for the full raster so every covered tree can be sampled.
with rasterio.open(landsat_path) as landsat:
    full_bands = landsat.read([4, 5], masked=True)
    full_ndvi = calculate_ndvi(full_bands[0], full_bands[1])
    full_transform = landsat.transform

# For a point, the median is just the value of the pixel containing that point.
tree_stats = zonal_stats(
    street_trees, full_ndvi, affine=full_transform,
    stats=["median"], nodata=np.nan,
)
street_trees["NDVI"] = [result["median"] for result in tree_stats]
print("Trees without a usable NDVI:", street_trees["NDVI"].isna().sum())
street_trees[["objectid", "NDVI"]].head()

### 2.2.3 Plotting the results

Make two plots of the results:

1. A histogram of the NDVI values, using matplotlib's `hist` function. Include a vertical line that marks the NDVI = 0 threshold
1. A plot of the street tree points, colored by the NDVI value, using geopandas' `plot` function. Include the city limits boundary on your plot.

The figures should be clear and well-styled, with for example, labels for axes, legends, and clear color choices. 

**How to read these plots:** The histogram shows the spread of values; the dashed line marks zero. The map shows where those values occur. Gray points, if present, have no usable value.

In [ ]:
valid_tree_ndvi = street_trees["NDVI"].dropna()
fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)

# Histogram: count how many tree locations fall into each NDVI range.
axes[0].hist(valid_tree_ndvi, bins=30, color="#31856b", edgecolor="white")
axes[0].axvline(0, color="#b3261e", linestyle="--", linewidth=2, label="NDVI = 0")
axes[0].set_title("NDVI at street tree locations")
axes[0].set_xlabel("NDVI")
axes[0].set_ylabel("Number of tree points")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.2)

# Map: color each tree point by the value of its satellite pixel.
city_limits.boundary.plot(ax=axes[1], color="#444444", linewidth=0.8)
street_trees.plot(
    ax=axes[1], column="NDVI", cmap="RdYlGn", vmin=-1, vmax=1,
    markersize=9, alpha=0.85, legend=True,
    legend_kwds={"label": "NDVI", "shrink": 0.75},
    missing_kwds={"color": "gray"},
)
axes[1].set_title("Tree points colored by NDVI")
axes[1].set_aspect("equal")
axes[1].set_axis_off()
plt.show()

print(f"Median NDVI at tree points: {valid_tree_ndvi.median():.3f}")
print(f"Points with NDVI below zero: {(valid_tree_ndvi < 0).sum():,}")

**Interpretation:** One pixel covers 30 × 30 meters, so it can include a tree, a road, a building, and grass together. A low value at a tree point does not prove the tree is unhealthy or missing. This is vegetation information about the surrounding pixel, not a measurement of one tree's health.